# **Assignment 02 on Training CNNs**

### **Foundational Setup and Training Statics Calculation** 
This code handles fixing the random seeds for reproducibility, calculating the exact channel means and standard deviations from the raw CIFAR-10 training data, applying those statistics to normalize the dataset, and splitting it into the required 45k/5k subsets.

In [4]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader
import numpy as np
import random

# 1. Fix random seeds for complete reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
root_dir = './data'

# 2. Load raw CIFAR-10 strictly to get the indices for the split
raw_train_set = torchvision.datasets.CIFAR10(root=root_dir, train=True, download=False)

# Split into 45k and 5k to get the exact indices for the training subset
train_size = 45000
val_size = 5000
raw_train_subset, raw_val_subset = random_split(
    raw_train_set, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

# 3. Extract ONLY the 45k training images to calculate statistics
# raw_train_subset.indices safely extracts just the 45,000 training images
train_45k_data = raw_train_set.data[raw_train_subset.indices]

cifar_mean = train_45k_data.mean(axis=(0, 1, 2)) / 255.0
cifar_std = train_45k_data.std(axis=(0, 1, 2)) / 255.0

print(f"Calculated 45k Training Mean (RGB): {cifar_mean}")
print(f"Calculated 45k Training Std (RGB): {cifar_std}")

# 4. Define the transform using ONLY the 45k computed statistics
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean.tolist(), std=cifar_std.tolist())
])

# 5. Load the datasets again, this time with the normalization transform applied
full_train_dataset_transformed = torchvision.datasets.CIFAR10(root=root_dir, train=True, download=False, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root=root_dir, train=False, download=False, transform=transform)

# 6. Re-apply the EXACT SAME split (using the same seed) so our transformed datasets match the indices
train_dataset, val_dataset = random_split(
    full_train_dataset_transformed, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42) 
)

# 7. Initialize DataLoaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train set size: {len(train_dataset)} | Val set size: {len(val_dataset)} | Test set size: {len(test_dataset)}")

Calculated 45k Training Mean (RGB): [0.49165571 0.48233405 0.44667923]
Calculated 45k Training Std (RGB): [0.24708116 0.24350869 0.26158066]
Train set size: 45000 | Val set size: 5000 | Test set size: 10000


## **Task 01: Backpropagation**  

### **(a) Implement Linear, ReLU, and softmax cross-entropy with explicit forward/backward (no autograd), and assemble an MLP**

### 1. Mathematical Formulation for Forward and Backward Passes

**Linear Layer** ($Y = XW + b$)
*   **Forward:** Given input $X \in \mathbb{R}^{N \times D_{in}}$, weights $W \in \mathbb{R}^{D_{in} \times D_{out}}$, and bias $b \in \mathbb{R}^{D_{out}}$. 
    $$Y = XW + b$$
*   **Backward:** Given the upstream gradient $dY \in \mathbb{R}^{N \times D_{out}}$.
    $$dW = X^T dY$$
    $$db = \sum_{i=1}^{N} dY_i$$
    $$dX = dY W^T$$

**ReLU Activation** ($Y = \max(0, X)$)
*   **Forward:** 
    $$Y = \max(0, X)$$
*   **Backward:** The gradient passes through only where the original input was strictly positive. Let $\mathbb{I}$ be the indicator function and $\odot$ be the element-wise product.
    $$dX = dY \odot \mathbb{I}(X > 0)$$

**Softmax Cross-Entropy**
*   **Forward:** We subtract the maximum value per row for numerical stability before exponentiating.
    $$P_{i,j} = \frac{e^{X_{i,j} - \max_k(X_{i,k})}}{\sum_k e^{X_{i,k} - \max_k(X_{i,k})}}$$
    $$L = -\frac{1}{N} \sum_{i=1}^{N} \log(P_{i, y_i})$$
*   **Backward:** The gradient of the combined softmax and cross-entropy loss with respect to the input logits $X$ simplifies cleanly. Let $Y_{true}$ be the one-hot encoded targets.
    $$dX = \frac{1}{N} (P - Y_{true})$$

In [5]:
import torch

class Linear:
    def __init__(self, in_features, out_features):
        # Kaiming initialization for weights
        self.W = torch.randn(in_features, out_features) * (2.0 / in_features)**0.5
        self.b = torch.zeros(out_features)
        
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = dout.sum(dim=0)
        dx = dout @ self.W.T
        return dx

class ReLU:
    def __init__(self):
        self.x = None

    def forward(self, x):
        self.x = x
        return torch.clamp(x, min=0)

    def backward(self, dout):
        dx = dout.clone()
        dx[self.x <= 0] = 0
        return dx

class SoftmaxCrossEntropy:
    def __init__(self):
        self.probs = None
        self.targets = None

    def forward(self, logits, targets):
        self.targets = targets
        N = logits.shape[0]
        
        # Shift logits for numerical stability
        shifted_logits = logits - logits.max(dim=1, keepdim=True).values
        exp_logits = torch.exp(shifted_logits)
        self.probs = exp_logits / exp_logits.sum(dim=1, keepdim=True)
        
        # Cross-entropy loss
        corect_log_probs = -torch.log(self.probs[torch.arange(N), targets] + 1e-9)
        loss = corect_log_probs.sum() / N
        return loss

    def backward(self):
        N = self.probs.shape[0]
        dx = self.probs.clone()
        dx[torch.arange(N), self.targets] -= 1
        dx /= N
        return dx

class MLP:
    def __init__(self, layer_sizes):
        self.layers = []
        for i in range(len(layer_sizes) - 1):
            self.layers.append(Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:
                self.layers.append(ReLU())
        self.criterion = SoftmaxCrossEntropy()

    def forward(self, x):
        out = x
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self):
        dout = self.criterion.backward()
        for layer in reversed(self.layers):
            dout = layer.backward(dout)